In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import mokapot

import pyXLMS
from pyXLMS.data import create_csm
from pyXLMS.parser.util import format_sequence
from pyXLMS.parser.util import get_bool_from_value

from typing import List
from typing import Dict
from typing import Any
from typing import Literal

plt.style.use("seaborn-v0_8")

In [3]:
df = pd.read_csv(
    "data/THIDDIAXL003_DIAmethodEval_SN20c4_Report_FM_crosslinking_plusDecoy_req_DIA12_CV486075.csv_annotated.csv_grouped_by_residue_pair.csv",
    low_memory=False,
)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14492 entries, 0 to 14491
Data columns (total 73 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   R.FileName                            14492 non-null  str    
 1   R.Condition                           14492 non-null  str    
 2   PG.ProteinNames                       14492 non-null  str    
 3   PG.Cscore                             3623 non-null   float64
 4   EG.Library                            14492 non-null  str    
 5   EG.PrecursorId                        14492 non-null  str    
 6   EG.Cscore                             14492 non-null  float64
 7   FG.Charge                             14492 non-null  int64  
 8   FG.Comment                            14492 non-null  str    
 9   F.CalibratedMz                        14492 non-null  float64
 10  PG.Pvalue                             14492 non-null  float64
 11  PG.PValue (Run-Wise)      

In [5]:
df["MP.Target"] = df.apply(lambda row: not (row["PP.IsDecoyA"] or row["PP.IsDecoyB"]), axis=1)

In [8]:
df["MK.Peptide"] = df.apply(lambda row: row["PP.PeptideA"]+row["PP.PeptideB"], axis=1)

In [10]:
df["MK.Protein"] = df.apply(lambda row: ";".join(set(row["PP.ProteinA"].split(";")) | set(row["PP.ProteinB"].split(";"))), axis=1)

In [11]:
rescoring_features = [  
    "PG.Cscore",  
    "EG.Cscore",
    "FG.Charge",    
    "PG.Pvalue",
    "PG.PValue (Run-Wise)",
    "PG.Qvalue",
    "PG.QValue (Run-Wise)",
    "EG.GlobalPrecursorQvalue",
    "EG.MaxChannelQvalue",
    "EG.MinChannelQvalue",
    "EG.Qvalue",
    "EG.InSourceFragmentationParentQvalue",
    "EG.AvgProfileQvalue",
    "EG.MaxProfileQvalue",
    "EG.MinProfileQvalue",
    "EG.PercentileQvalue",
    "FG.Qvalue",
    "PP.MatchedIonsA",  
    "PP.TotalIonsA", 
    "PP.MatchedIonsB",  
    "PP.TotalIonsB", 
    "PP.RelativeMatchScoreA",
    "PP.RelativeMatchScoreB",
    "PP.PartialCscoreA",
    "PP.PartialCscoreB",
    "PP.CompositeRelativeMatchScore",
    "PP.CompositePartialCscore",          
    "PP.IsDecoyA",   
    "PP.IsDecoyB",    
    "PP.SequenceCoverageNTermAlpha",
    "PP.SequenceCoverageNTermBeta",
    "PP.SequenceCoverageNTermFull",
    "PP.SequenceCoverageCTermAlpha",
    "PP.SequenceCoverageCTermBeta",
    "PP.SequenceCoverageCTermFull",
    "PP.SequenceCoverageAlpha",
    "PP.SequenceCoverageBeta",
    "PP.SequenceCoverageFull",
    "PP.UniScoreAlpha",  
    "PP.UniScoreBeta", 
    "PP.UniScoreFull",
    "PP.PepLenAlpha",  
    "PP.PepLenBeta",  
    "PP.NumberCrosslinkFragmentsAlpha", 
    "PP.NumberCrosslinkFragmentsBeta",  
    "PP.NumberCrosslinkFragmentsFull",  
    "PP.NormalizedCrosslinkFragmentsAlpha",
    "PP.NormalizedCrosslinkFragmentsBeta",
    "PP.NormalizedCrosslinkFragmentsFull",
]

In [12]:
# https://mokapot.readthedocs.io/en/latest/api/dataset.html
psms = mokapot.dataset.LinearPsmDataset(
    psms=df,
    target_column="MP.Target",
    spectrum_columns=["R.FileName", "R.Condition", "PP.PseudoScanNumber"],
    peptide_column="MP.Peptide",
    protein_column="MP.Protein",
    feature_columns=rescoring_features,
    copy_data=True,
    rng=1337,
)

Missing values detected in the following features:
  - PG.Cscore
  - EG.GlobalPrecursorQvalue
  - EG.MaxChannelQvalue
  - EG.MinChannelQvalue
  - EG.Qvalue
  - EG.InSourceFragmentationParentQvalue
  - EG.AvgProfileQvalue
  - EG.MaxProfileQvalue
  - EG.MinProfileQvalue
  - EG.PercentileQvalue
  - FG.Qvalue
Dropping features with missing values...


In [13]:
results, models = mokapot.brew(psms)

AttributeError: `np.float_` was removed in the NumPy 2.0 release. Use `np.float64` instead.